# 02. Segmentación clásica del ventrículo lateral

Pipeline sin aprendizaje:

1. Normalización de intensidad
2. Máscara cerebral (skull stripping)
3. Umbral de Otsu: LCR oscuro en T1
4. Erosión del cerebro para excluir LCR externo
5. Componentes conexas centrales
6. Crecimiento de región y relleno de huecos
7. Evaluación contra el experto (Dice, error de volumen)

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

from src.io import list_scans, load, load_seg
from src.paths import OUTPUTS
from src.seg_clasica import otsu, skull_strip, dice

GT_VENT = [4]
SS_VENT = [4, 5, 43, 44]
scans = list_scans()
len(scans)

## Caso de ejemplo

In [ ]:
CASO = "sub_042"
s = next((x for x in scans if x.parent.name == CASO), scans[len(scans) // 2])
img, zooms = load(s / "eT1W_FFE_SVR.nii.gz")
gt, _ = load_seg(s / "segmentation" / "gt.nii.gz")
g = np.isin(gt, GT_VENT)
k = int(np.argmax(g.sum(axis=(0, 1))))
ml = float(np.prod(zooms)) / 1000
print(img.shape, zooms, "corte", k, "| ventrículo experto", round(g.sum() * ml, 1), "mL")

def show(ax, im, title, mask=None, color="teal", cmap="gray"):
    ax.imshow(np.rot90(im[:, :, k]), cmap=cmap)
    if mask is not None:
        m = np.rot90(mask[:, :, k])
        ax.contourf(m, levels=[0.5, 1.5], colors=[color], alpha=0.45)
        ax.contour(m, levels=[0.5], colors=[color], linewidths=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

## 1-2. Normalización y máscara cerebral
El fondo se separa con Otsu; una erosión desconecta el cráneo y nos quedamos con el componente más grande.

In [ ]:
brain_classic = skull_strip(img, zooms)
brain_ss = ndi.binary_fill_holes(load_seg(s / "segmentation" / "synthseg.nii.gz")[0] > 0)

fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
show(ax[0], img, "SVR original")
show(ax[1], img, "Máscara clásica", brain_classic, "gold")
show(ax[2], img, "Máscara SynthSeg (referencia)", brain_ss, "gold")
plt.tight_layout(); plt.show()
print("Dice máscara clásica vs SynthSeg:", round(dice(brain_classic, brain_ss), 3))

In [ ]:
BRAIN = "clasica"
brain = brain_classic if BRAIN == "clasica" else brain_ss

vals = img[brain]
lo, hi = np.percentile(vals, [1, 99])
norm = np.clip((img - lo) / (hi - lo), 0, 1)
t = otsu(norm[brain])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(norm[brain].ravel(), bins=200, color="gray")
ax[0].axvline(t, color="red", label=f"Otsu = {t:.2f}")
ax[0].set_xlabel("intensidad normalizada"); ax[0].legend()
ax[0].set_title("Histograma dentro del cerebro")
dark = (norm < t) & brain
show(ax[1], norm, "3. Oscuro (< Otsu)", dark, "red")
plt.tight_layout(); plt.show()

## 4-6. Erosión, componentes centrales y crecimiento

In [ ]:
ball = np.ones((3, 3, 3), bool)
r = max(1, int(round(10 / min(zooms))))
inner = ndi.binary_erosion(brain, ball, iterations=r)
cand = ndi.binary_opening(dark & inner, ball)

lab, n = ndi.label(cand)
if n == 0:
    raise RuntimeError("No hay candidatos: baja EROSION (10 mm) o revisa la máscara cerebral (prueba BRAIN = 'synthseg')")
sizes = ndi.sum(cand, lab, range(1, n + 1)) * ml
center = np.array(ndi.center_of_mass(brain))
coms = np.array(ndi.center_of_mass(cand, lab, range(1, n + 1)))
dist = np.linalg.norm((coms - center) * np.array(zooms), axis=1)
keep = [i + 1 for i in range(n) if sizes[i] >= 0.5 and dist[i] < 45] or [int(np.argmax(sizes)) + 1]
seeds = np.isin(lab, keep)

shell = ndi.binary_erosion(brain, ball, iterations=max(1, int(round(3 / min(zooms)))))
grown = ndi.binary_propagation(seeds, mask=dark & shell)
final = ndi.binary_closing(grown, ball, iterations=2)
for z in range(final.shape[2]):
    final[:, :, z] = ndi.binary_fill_holes(final[:, :, z])
final &= brain

print(f"componentes: {n} | conservados: {len(keep)}")
fig, ax = plt.subplots(1, 4, figsize=(17, 4.5))
show(ax[0], img, "4. Oscuro dentro del cerebro erosionado", cand, "red")
show(ax[1], img, "5. Semillas centrales", seeds, "orange")
show(ax[2], img, "6. Crecimiento + relleno", final, "teal")
show(ax[3], img, "Experto", g, "purple")
plt.tight_layout(); plt.show()

## 7. Evaluación del caso

In [ ]:
comp = np.zeros(g.shape[:2] + (4,))
gk, pk = g[:, :, k], final[:, :, k]
comp[gk & pk] = (0.1, 0.8, 0.2, 0.75)
comp[~gk & pk] = (0.9, 0.1, 0.1, 0.75)
comp[gk & ~pk] = (0.1, 0.4, 1.0, 0.75)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(np.rot90(img[:, :, k]), cmap="gray")
ax.imshow(np.rot90(comp))
ax.axis("off")
err = (final.sum() - g.sum()) / g.sum() * 100
ax.set_title(f"Dice {dice(g, final):.3f} | error vol {err:+.1f}%\nverde acierto, rojo sobra, azul falta")
plt.show()

## Todos los casos

In [ ]:
from src.seg_clasica import segment

rows = []
for s_ in scans:
    im, z = load(s_ / "eT1W_FFE_SVR.nii.gz")
    gt_, _ = load_seg(s_ / "segmentation" / "gt.nii.gz")
    g_ = np.isin(gt_, GT_VENT)
    b = skull_strip(im, z) if BRAIN == "clasica" else ndi.binary_fill_holes(load_seg(s_ / "segmentation" / "synthseg.nii.gz")[0] > 0)
    p = segment(im, b, z)
    ss_ = np.isin(load_seg(s_ / "segmentation" / "synthseg.nii.gz")[0], SS_VENT)
    m_ = float(np.prod(z)) / 1000
    rows.append({"subject": s_.parent.name, "scan": s_.name, "vol_gt": g_.sum() * m_, "vol_clasico": p.sum() * m_,
                 "dice_clasico": dice(g_, p), "dice_synthseg": dice(g_, ss_)})
    print(f"{s_.parent.name}: {rows[-1]['dice_clasico']:.2f}")
res = pd.DataFrame(rows).sort_values("vol_gt")
res.to_csv(OUTPUTS / "seg_clasica_resultados.csv", index=False)
res[["dice_clasico", "dice_synthseg"]].describe().round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(res.vol_gt, res.dice_synthseg, color="darkorange", label="SynthSeg")
ax[0].scatter(res.vol_gt, res.dice_clasico, color="teal", label="Clásico")
ax[0].set_xlabel("volumen experto (mL)"); ax[0].set_ylabel("Dice"); ax[0].set_ylim(0, 1); ax[0].legend()
ax[0].set_title("Dice según severidad")
lim = [0, max(res.vol_gt.max(), res.vol_clasico.max()) * 1.05]
ax[1].scatter(res.vol_gt, res.vol_clasico, color="teal")
ax[1].plot(lim, lim, "k--", lw=1)
ax[1].set_xlabel("volumen experto (mL)"); ax[1].set_ylabel("volumen clásico (mL)")
ax[1].set_title("Concordancia de volumen")
plt.tight_layout(); plt.savefig(OUTPUTS / "clasico_resumen.png", dpi=150); plt.show()
res.sort_values("dice_clasico").head(5)

## Notas

- Peores casos: revisar si son lactantes (contraste invertido) o si el ventrículo toca la corteza.
- Parámetros a ajustar: `EROSION_MM` (10), distancia al centro (45 mm), radio del borde (3 mm).
- `BRAIN = "synthseg"` usa la máscara de SynthSeg para aislar el error del skull stripping.